# 06 Predictions: hidden-link check and new candidates

Uses the models of all 5 folds (R-GCN + fixes and two-tower ESM-2 for C-T, R-GCN + fixes for H-C):
- **hidden-link check**: each fold's models never saw that fold's test links; for every test compound, how many of its top 10 predicted
  targets are those hidden links, compared with what random guessing would give
- **new candidates**: the 5 fold models are averaged (each was trained on a different 65% of the links, so together they use all of them),
  and every candidate gets a count of how many of the 5 fold models put it in their own top 10. A candidate found by 5 of 5 is more
  reliable than one found by 1 of 5.

Scores rank targets (or compounds) within one compound (or herb); they are not probabilities, so all lists are per compound and per herb.

Reads: dataset files for names, `prep/`, `results/metrics_rgcn.csv`
Writes: `results/hidden_link_check.csv`, `results/hidden_link_examples.csv`, `results/candidates_ct.csv`, `results/candidates_ht.csv`, `results/candidates_hc.csv`

## Setup

In [1]:
import hti
import numpy as np
import pandas as pd
import torch
from hti.data import DATA, Prep, RESULTS
from hti.metrics import herb_rows_to_ranks, herb_scores
from hti.models import row_ranks
from hti.train import train_graph_model, train_two_tower

prep = Prep()
herbs = pd.read_csv(DATA / 'herbs.tsv', sep='\t', dtype=str).set_index('herb_id')
compounds = pd.read_csv(DATA / 'compounds.tsv', sep='\t', dtype=str).set_index('compound_id')
targets = pd.read_csv(DATA / 'target_features_aac.csv').set_index('protein_id')
symbols = pd.read_csv(DATA / 'compound_target_labeled.csv', dtype=str).drop_duplicates('protein_id').set_index('protein_id')['hgnc_symbol']

herb_name = [herbs.at[h, 'botanical_name'] for h in prep.herb_ids]
compound_name = [compounds.at[c, 'Phytochemical name_standardised'] for c in prep.compound_ids]
target_symbol = [symbols.get(t, t) for t in prep.target_ids]
target_name = [f"{symbols.get(t, '')} ({targets.at[t, 'protein_name']})" for t in prep.target_ids]

## Train the models of all 5 folds
For C-T the R-GCN + fixes and two-tower ESM-2 scores are averaged by rank within each compound. About 4 to 5 minutes on an RTX 4060.

In [2]:
ct_scores, ct_folds, hc_scores = [], [], []
for k in range(prep.k):
    fold = prep.fold('ct_edge', k)
    rgcn, _, _ = train_graph_model(prep, fold, 'ct_edge', config='fixes', seed=k)
    tower, _, _ = train_two_tower(prep, fold, target_feature='esm', seed=k)
    ct_scores.append(0.5 * row_ranks(rgcn) + 0.5 * row_ranks(tower))
    ct_folds.append(fold)

    rgcn_hc, _, _ = train_graph_model(prep, prep.fold('hc_edge', k), 'hc_edge', config='fixes', seed=k)
    hc_scores.append(row_ranks(rgcn_hc))
    print('fold', k, 'done')

fold 0 done
fold 1 done
fold 2 done
fold 3 done
fold 4 done


## Hidden-link check
For every test compound of a fold, its top 10 targets are taken from that fold's model, leaving out its training links.
Each target in the top 10 is a hidden test link, a validation link, or unlabeled. "Expected by chance" is how many test links a random
top 10 would contain (10 × the compound's test links ÷ the targets it can be ranked against).

In [3]:
rows, examples = [], []
for k, (scores, fold) in enumerate(zip(ct_scores, ct_folds)):
    status = torch.zeros(prep.n_compounds, prep.n_targets, dtype=torch.int8, device=prep.device)   # 0 unlabeled
    for code, part in [(1, 'train'), (2, 'val'), (3, 'test')]:
        p = fold[part]['pos']
        status[p[0], p[1]] = code
    s = scores.clone()
    s[status == 1] = -1                                   # training links are not predictions
    test_compounds = torch.unique(fold['test']['pos'][0])
    top = torch.topk(s[test_compounds], 10, dim=1).indices
    in_top = status[test_compounds[:, None], top]
    n_test = (status[test_compounds] == 3).sum(1).float()
    n_cand = (status[test_compounds] != 1).sum(1).float()
    rows.append({'fold': k, 'test compounds': len(test_compounds),
                 'test hits in top 10': float((in_top == 3).sum(1).float().mean()),
                 'expected by chance': float((10 * n_test / n_cand).mean()),
                 'validation links in top 10': float((in_top == 2).sum(1).float().mean()),
                 'unlabeled in top 10': float((in_top == 0).sum(1).float().mean()),
                 'compounds with a test hit in top 10': float(((in_top == 3).sum(1) > 0).float().mean())})
    if k == 0:
        busiest = test_compounds[torch.argsort(n_test, descending=True)[:3]]
        for c in busiest.tolist():
            t10 = torch.topk(s[c], 10).indices.tolist()
            st = status[c, t10].tolist()
            examples.append({'compound': compound_name[c], 'hidden test targets': int((status[c] == 3).sum()),
                             'test hits in top 10': st.count(3), 'validation links in top 10': st.count(2),
                             'unlabeled in top 10': st.count(0),
                             'test hits found': ', '.join(target_symbol[t] for t, x in zip(t10, st) if x == 3)})

check = pd.DataFrame(rows).set_index('fold')
check.loc['mean'] = check.mean()
check.to_csv(RESULTS / 'hidden_link_check.csv')
pd.DataFrame(examples).to_csv(RESULTS / 'hidden_link_examples.csv', index=False)
display(check.round(3))
pd.DataFrame(examples)

,test compounds,test hits in top 10,expected by chance,validation links in top 10,unlabeled in top 10,compounds with a test hit in top 10
fold,,,,,,
0,470.0,1.757,0.047,1.096,7.147,0.679
1,456.0,1.774,0.048,1.134,7.092,0.693
2,463.0,1.654,0.048,1.168,7.177,0.659
3,459.0,1.739,0.048,1.231,7.031,0.699
4,459.0,1.673,0.048,1.190,7.137,0.656
mean,461.4,1.720,0.048,1.164,7.117,0.677


,compound,hidden test targets,test hits in top 10,validation links in top 10,unlabeled in top 10,test hits found
0,Quercetin,79,7,2,1,"MAPK3, NPY1R, HTR2A, SLC6A2, AKR1B1, CYP1A2, C..."
1,Apigenin,58,5,5,0,"PIM1, ATAD5, SLCO2B1, GABRG3, AURKB"
2,Genistein,51,5,5,0,"SLC6A3, CYP2E1, PDE5A, OPRM1, CYP2C19"


## Average the 5 fold models and count agreement
For every compound (or herb), each fold model gives its own top 10 new targets (or compounds). "folds in top 10" counts how many of the
5 fold models agree on a candidate.

In [4]:
def top10_mask(scores, exclude):
    s = scores.clone()
    s[exclude] = -1e9
    mask = torch.zeros(exclude.shape, dtype=torch.int32, device=exclude.device)
    mask.scatter_(1, torch.topk(s, 10, dim=1).indices, 1)
    return mask

ct_mean = torch.stack(ct_scores).mean(0)
ct_votes = sum(top10_mask(s, prep.known['ct']) for s in ct_scores)

rgcn_metrics = pd.read_csv(RESULTS / 'metrics_rgcn.csv')
how = rgcn_metrics[(rgcn_metrics.task == 'ct_edge') & (rgcn_metrics.model == 'R-GCN + fixes') &
                   (rgcn_metrics.set == 'test')]['HT aggregation (model only)'].mode()[0]
herb_fold_scores = [herb_rows_to_ranks(herb_scores(s, prep.herb_compounds, how)) for s in ct_scores]
ht_mean = torch.stack(herb_fold_scores).mean(0)
ht_votes = sum(top10_mask(s, prep.ht_known) for s in herb_fold_scores)

hc_mean = torch.stack(hc_scores).mean(0)
hc_votes = sum(top10_mask(s, prep.known['hc']) for s in hc_scores)
print('herb aggregation:', how)

herb aggregation: max


## New C-T candidates
The 5 best targets for every compound that are not a known link. `known targets` is how many known targets the compound has; for
compounds with none, the prediction comes from chemistry alone.

In [5]:
s = ct_mean.clone()
s[prep.known['ct']] = -1
top = torch.topk(s, 5, dim=1)
n_known = prep.known['ct'].sum(1).cpu().numpy()
rows = []
for c in range(prep.n_compounds):
    for rank, (t, score) in enumerate(zip(top.indices[c].tolist(), top.values[c].tolist()), start=1):
        rows.append({'compound': compound_name[c], 'compound_id': prep.compound_ids[c], 'known targets': int(n_known[c]), 'rank': rank,
                     'target': target_name[t], 'protein_id': prep.target_ids[t], 'score': round(score, 4),
                     'folds in top 10': int(ct_votes[c, t])})
candidates_ct = pd.DataFrame(rows)
candidates_ct.to_csv(RESULTS / 'candidates_ct.csv', index=False)
candidates_ct[candidates_ct['compound'].isin(['Quercetin', 'Curcumin', 'Withaferin A'])]

,compound,compound_id,known targets,rank,target,protein_id,score,folds in top 10
30,Quercetin,IMPPAT3_PHYID000007,354,1,NFE2L2 (Nuclear factor erythroid 2-related fac...,UNIPROT:Q16236,0.8867,4
31,Quercetin,IMPPAT3_PHYID000007,354,2,FYN (Tyrosine-protein kinase Fyn),UNIPROT:P06241,0.8628,4
32,Quercetin,IMPPAT3_PHYID000007,354,3,ERBB2 (Receptor tyrosine-protein kinase erbB-2),UNIPROT:P04626,0.8513,4
33,Quercetin,IMPPAT3_PHYID000007,354,4,IDH1 (Isocitrate dehydrogenase [NADP] cytoplas...,UNIPROT:O75874,0.8371,2
34,Quercetin,IMPPAT3_PHYID000007,354,5,ATXN2 (Ataxin-2),UNIPROT:Q99700,0.8269,2
6970,Curcumin,IMPPAT3_PHYID005594,193,1,ATXN2 (Ataxin-2),UNIPROT:Q99700,0.9084,4
6971,Curcumin,IMPPAT3_PHYID005594,193,2,AKR1B1 (Aldo-keto reductase family 1 member B1),UNIPROT:P15121,0.9064,4
6972,Curcumin,IMPPAT3_PHYID005594,193,3,SLCO1B1 (Solute carrier organic anion transpor...,UNIPROT:Q9Y6L6,0.9042,3
6973,Curcumin,IMPPAT3_PHYID005594,193,4,PTPN1 (Tyrosine-protein phosphatase non-recept...,UNIPROT:P18031,0.9022,4
6974,Curcumin,IMPPAT3_PHYID005594,193,5,CBX1 (Chromobox protein homolog 1),UNIPROT:P83916,0.8952,4


## New H-T candidates
The 10 best targets for every herb, among herb-target pairs with no path through any known link. Herb scores use the model-only rule
picked most often on validation in 03 (no popularity blend), so the list shows what the models themselves suggest.

In [6]:
s = ht_mean.clone()
s[prep.ht_known] = -1e9
top = torch.topk(s, 10, dim=1)
rows = []
for h in range(prep.n_herbs):
    for rank, t in enumerate(top.indices[h].tolist(), start=1):
        rows.append({'herb': herb_name[h], 'compounds in herb': len(prep.herb_compounds[h]), 'rank': rank,
                     'target': target_name[t], 'protein_id': prep.target_ids[t], 'aggregation': how,
                     'folds in top 10': int(ht_votes[h, t])})
candidates_ht = pd.DataFrame(rows)
candidates_ht.to_csv(RESULTS / 'candidates_ht.csv', index=False)
candidates_ht[candidates_ht['herb'].isin(['Withania somnifera', 'Curcuma longa'])]

,herb,compounds in herb,rank,target,protein_id,aggregation,folds in top 10
190,Curcuma longa,189,1,GABRB3 (Gamma-aminobutyric acid receptor subun...,UNIPROT:P28472,max,5
191,Curcuma longa,189,2,UGT1A7 (UDP-glucuronosyltransferase 1A7),UNIPROT:Q9HAW7,max,5
192,Curcuma longa,189,3,GABRG1 (Gamma-aminobutyric acid receptor subun...,UNIPROT:Q8N1C3,max,5
193,Curcuma longa,189,4,GABRB1 (Gamma-aminobutyric acid receptor subun...,UNIPROT:P18505,max,4
194,Curcuma longa,189,5,PTPN6 (Tyrosine-protein phosphatase non-recept...,UNIPROT:P29350,max,4
195,Curcuma longa,189,6,GABRA3 (Gamma-aminobutyric acid receptor subun...,UNIPROT:P34903,max,5
196,Curcuma longa,189,7,PTPN11 (Tyrosine-protein phosphatase non-recep...,UNIPROT:Q06124,max,4
197,Curcuma longa,189,8,GABRA5 (Gamma-aminobutyric acid receptor subun...,UNIPROT:P31644,max,3
198,Curcuma longa,189,9,PTPRE (Receptor-type tyrosine-protein phosphat...,UNIPROT:P23469,max,4
199,Curcuma longa,189,10,DUSP3 (Dual specificity protein phosphatase 3),UNIPROT:P51452,max,3


## New H-C candidates
The 10 compounds the models rank highest for every herb, where the data has no link.

In [7]:
s = hc_mean.clone()
s[prep.known['hc']] = -1e9
top = torch.topk(s, 10, dim=1)
rows = []
for h in range(prep.n_herbs):
    for rank, c in enumerate(top.indices[h].tolist(), start=1):
        rows.append({'herb': herb_name[h], 'rank': rank, 'compound': compound_name[c], 'compound_id': prep.compound_ids[c],
                     'score': round(float(s[h, c]), 4), 'folds in top 10': int(hc_votes[h, c])})
candidates_hc = pd.DataFrame(rows)
candidates_hc.to_csv(RESULTS / 'candidates_hc.csv', index=False)
candidates_hc[candidates_hc['herb'].isin(['Withania somnifera', 'Curcuma longa'])]

,herb,rank,compound,compound_id,score,folds in top 10
190,Curcuma longa,1,alpha-Copaene,IMPPAT3_PHYID000022,0.9819,3
191,Curcuma longa,2,Elemol,IMPPAT3_PHYID000682,0.9810,1
192,Curcuma longa,3,(+)-gamma-Cadinene,IMPPAT3_PHYID000235,0.9806,2
193,Curcuma longa,4,gamma-Eudesmol,IMPPAT3_PHYID000607,0.9776,1
194,Curcuma longa,5,gamma-Muurolene,IMPPAT3_PHYID000239,0.9769,1
195,Curcuma longa,6,(-)-alpha-Cadinol,IMPPAT3_PHYID000438,0.9751,2
196,Curcuma longa,7,(+)-gamma-Gurjunene,IMPPAT3_PHYID000605,0.9747,0
197,Curcuma longa,8,Sabinene hydrate,IMPPAT3_PHYID000617,0.9724,1
198,Curcuma longa,9,Nerol,IMPPAT3_PHYID000417,0.9720,2
199,Curcuma longa,10,beta-Selinene,IMPPAT3_PHYID000241,0.9694,1


## How stable are the candidates?
Share of candidates that 5 of 5, and at least 3 of 5, fold models agree on.

In [8]:
pd.DataFrame([{'list': name, 'candidates': len(d),
               'found by 5 of 5 folds': round((d['folds in top 10'] == 5).mean(), 3),
               'found by at least 3 of 5': round((d['folds in top 10'] >= 3).mean(), 3)}
              for name, d in [('C-T', candidates_ct), ('H-T', candidates_ht), ('H-C', candidates_hc)]])

,list,candidates,found by 5 of 5 folds,found by at least 3 of 5
0,C-T,11595,0.290,0.846
1,H-T,620,0.194,0.600
2,H-C,620,0.019,0.326
